# Análise Exploratória — Dados ANAC
Projeto: Panorama da Aviação Doméstica Brasileira (2016–2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'

## 1. Dataset de Produção Aeronáutica

In [ ]:
prod_path = RAW / 'producao' / 'Dados_Estatisticos.csv'
df = pd.read_csv(prod_path, encoding='latin1', sep=';', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Tipos e nulos
info = pd.DataFrame({
    'dtype': df.dtypes,
    'nulos': df.isnull().sum(),
    'nulos_%': (df.isnull().mean() * 100).round(2),
    'únicos': df.nunique()
})
info

In [ ]:
# Nomes reais das colunas
print(df.columns.tolist())

In [ ]:
# Intervalo de datas
# Detectar colunas de ano e mês
ano_col = [c for c in df.columns if 'ano' in c.lower() or 'year' in c.lower()][0]
mes_col = [c for c in df.columns if 'mes' in c.lower() or 'month' in c.lower() or 'mês' in c.lower()][0]
print(f'Ano: {df[ano_col].min()} – {df[ano_col].max()}')
print(f'Mês: {df[mes_col].unique()}')

In [ ]:
# Natureza dos voos
nat_col = [c for c in df.columns if 'natureza' in c.lower() or 'nature' in c.lower()][0]
df[nat_col].value_counts()

In [ ]:
# Filtrar apenas doméstico
df_dom = df[df[nat_col].str.contains('Dom', case=False, na=False)].copy()
print(f'Voos domésticos: {len(df_dom):,} linhas')

In [ ]:
# Empresas aéreas
emp_col = [c for c in df.columns if 'empresa' in c.lower() or 'compan' in c.lower() or 'sigla' in c.lower()][0]
print('Empresas presentes:')
df_dom[emp_col].value_counts().head(15)

In [ ]:
# Top 20 aeroportos por passageiros
pax_col = [c for c in df.columns if 'passageiro' in c.lower() or 'pax' in c.lower()][0]
aero_col = [c for c in df.columns if 'origem' in c.lower() and ('icao' in c.lower() or 'sigla' in c.lower() or 'aeroporto' in c.lower())]
print('Colunas aeroporto:', aero_col)

if aero_col:
    top20 = df_dom.groupby(aero_col[0])[pax_col].sum().sort_values(ascending=False).head(20)
    print(top20)

In [ ]:
# Série temporal: passageiros mensais (impacto COVID)
df_dom['periodo'] = df_dom[ano_col].astype(str) + '-' + df_dom[mes_col].astype(str).str.zfill(2)
serie = df_dom.groupby('periodo')[pax_col].sum().reset_index()
serie = serie[serie['periodo'] >= '2016-01']
print(serie.tail(10))

## 2. Dataset de Atrasos (Anexo I)

In [ ]:
# Carregar uma amostra do Anexo I (último ano disponível)
import glob
anexo1_files = sorted(glob.glob(str(RAW / 'atrasos' / '202*' / '*' / 'anexo_i.csv')))
print(f'Arquivos Anexo I encontrados: {len(anexo1_files)}')
if anexo1_files:
    a1 = pd.read_csv(anexo1_files[-1], encoding='latin1', sep=';', low_memory=False)
    print(f'Shape: {a1.shape}')
    print(a1.columns.tolist())
    a1.head(3)

In [ ]:
# Nulos Anexo I
if 'a1' in dir():
    pd.DataFrame({
        'dtype': a1.dtypes,
        'nulos_%': (a1.isnull().mean() * 100).round(2),
        'únicos': a1.nunique()
    })

In [ ]:
# Carregar Anexo II (consolidado por empresa + par de aeroportos)
anexo2_files = sorted(glob.glob(str(RAW / 'atrasos' / '202*' / '*' / 'anexo_ii.csv')))
if anexo2_files:
    a2 = pd.read_csv(anexo2_files[-1], encoding='latin1', sep=';', low_memory=False)
    print(f'Anexo II shape: {a2.shape}')
    print(a2.columns.tolist())
    a2.head(3)

In [ ]:
# Carregar Anexo III (consolidado por par de aeroportos)
anexo3_files = sorted(glob.glob(str(RAW / 'atrasos' / '202*' / '*' / 'anexo_iii.csv')))
if anexo3_files:
    a3 = pd.read_csv(anexo3_files[-1], encoding='latin1', sep=';', low_memory=False)
    print(f'Anexo III shape: {a3.shape}')
    print(a3.columns.tolist())
    a3.head(3)

## 3. Dataset de Aeródromos

In [ ]:
aero_path = RAW / 'aerodromos' / 'aerodromos_caracteristicas_gerais.csv'
aero = pd.read_csv(aero_path, encoding='latin1', sep=';', low_memory=False)
print(f'Shape: {aero.shape}')
print(aero.columns.tolist())
aero.head(5)

In [ ]:
# Verificar coordenadas lat/lon
geo_cols = [c for c in aero.columns if any(k in c.lower() for k in ['lat', 'lon', 'lng', 'coord'])]
print('Colunas geográficas:', geo_cols)
if geo_cols:
    print(aero[geo_cols].describe())

In [ ]:
# Distribuição por estado/região
uf_col = [c for c in aero.columns if 'uf' in c.lower() or 'estado' in c.lower() or 'state' in c.lower()]
if uf_col:
    print(aero[uf_col[0]].value_counts())

## 4. Resumo para os Slides

In [ ]:
print('=== RESUMO PARA SLIDES ===')
print(f'Produção: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'Período: {df[ano_col].min()} – {df[ano_col].max()}')
print(f'Empresas distintas: {df_dom[emp_col].nunique()}')
print(f'Arquivos Anexo I: {len(anexo1_files)}')
print(f'Aeródromos: {aero.shape[0]} registros')